# Fingerprints: build and evaluate a workload

Configure the workload below, run the notebook top-to-bottom, and inspect or save the resulting per-query measurements. Prefix and suffix queries use quantile bucket fingerprints; substring queries use the Arrow row-level bitmask scanner.

In [2]:
from pathlib import Path
import json
import subprocess
import sys
import time

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_DIR = Path.cwd()
INPUT_PATH = PROJECT_DIR / 'title_strs.parquet'
TEXT_COLUMN = 'title'

# These control only fingerprints needed by the workload below.
PREFIX_BITS = 8
SUBSTRING_BITS = 16
SUBSTRING_NGRAM = 1       # 1=characters (recommended), 2=bigrams
SUBSTRING_MAX_FREQ = 1.0  # See build_substr.py for the trade-off
BUILD_FINGERPRINTS = True # Set False to reuse existing artifacts
WARMUP_RUNS = 1
TIMED_REPETITIONS = 3
RESULTS_PATH = PROJECT_DIR / 'fingerprint_workload_results.csv'

assert INPUT_PATH.exists(), f'Missing input parquet: {INPUT_PATH}'
print(f'Project: {PROJECT_DIR}')
print(f'Input:   {INPUT_PATH}')

Project: /home/reese/row-level-string-fingerprints
Input:   /home/reese/row-level-string-fingerprints/title_strs.parquet


## Define the workload

Use `prefix`, `suffix`, or `substring` in `query_type`; `pattern` excludes SQL wildcard characters. Replace the example rows with your workload, or point `WORKLOAD_CSV` at a CSV containing `query_type,pattern` (an optional `name` column is retained in the report).

In [3]:
WORKLOAD_CSV = None  # e.g. PROJECT_DIR / 'my_workload.csv'

WORKLOAD = pd.DataFrame([
    {'name': 'prefix-the', 'query_type': 'prefix',    'pattern': 'the'},
    {'name': 'suffix-ing', 'query_type': 'suffix',    'pattern': 'ing'},
    {'name': 'contains-python', 'query_type': 'substring', 'pattern': 'python'},
])

if WORKLOAD_CSV is not None:
    WORKLOAD = pd.read_csv(WORKLOAD_CSV)

required_columns = {'query_type', 'pattern'}
missing = required_columns - set(WORKLOAD.columns)
assert not missing, f'Workload is missing columns: {sorted(missing)}'
WORKLOAD = WORKLOAD.copy()
WORKLOAD['query_type'] = WORKLOAD['query_type'].str.lower().str.strip()
WORKLOAD['pattern'] = WORKLOAD['pattern'].fillna('').astype(str)
assert WORKLOAD['query_type'].isin({'prefix', 'suffix', 'substring'}).all()
assert WORKLOAD['pattern'].ne('').all(), 'Patterns must not be empty.'
if 'name' not in WORKLOAD:
    WORKLOAD['name'] = WORKLOAD['query_type'] + ':' + WORKLOAD['pattern']

display(WORKLOAD)

,name,query_type,pattern
0,prefix-the,prefix,the
1,suffix-ing,suffix,ing
2,contains-python,substring,python


In [4]:
# Build exactly the artifacts this workload requires. The project builders write
# their standard output names into PROJECT_DIR. build_substr.py currently uses
# title_strs.parquet as its input, so keep INPUT_PATH at that standard location.
def run_builder(*args):
    print('+', ' '.join(map(str, args)))
    subprocess.run(args, cwd=PROJECT_DIR, check=True)

types = set(WORKLOAD['query_type'])
if BUILD_FINGERPRINTS:
    if 'prefix' in types:
        run_builder(sys.executable, 'build.py', '--bits', str(PREFIX_BITS), '--input', str(INPUT_PATH))
    if 'suffix' in types:
        run_builder(sys.executable, 'build.py', '--bits', str(PREFIX_BITS), '--suffix', '--input', str(INPUT_PATH))
    if 'substring' in types:
        if INPUT_PATH.name != 'title_strs.parquet':
            raise ValueError('build_substr.py requires the standard title_strs.parquet input name.')
        run_builder(sys.executable, 'build_substr.py', '--bits', str(SUBSTRING_BITS),
                    '--ngram', str(SUBSTRING_NGRAM), '--max-freq', str(SUBSTRING_MAX_FREQ))

expected = []
if 'prefix' in types:
    expected += [PROJECT_DIR / f'title_strs_prefix_b{PREFIX_BITS}.parquet', PROJECT_DIR / f'q{PREFIX_BITS}_prefix_boundaries.npy']
if 'suffix' in types:
    expected += [PROJECT_DIR / f'title_strs_suffix_b{PREFIX_BITS}.parquet', PROJECT_DIR / f'q{PREFIX_BITS}_suffix_boundaries.npy']
if 'substring' in types:
    expected += [PROJECT_DIR / 'title_strs_substr_fp16.parquet', PROJECT_DIR / 'substr_features.json']
missing = [str(path) for path in expected if not path.exists()]
assert not missing, 'Missing build artifacts:\n' + '\n'.join(missing)
print('Fingerprint artifacts are ready.')

+ /home/reese/row-level-string-fingerprints/.rowToBlock/bin/python build.py --bits 8 --input /home/reese/row-level-string-fingerprints/title_strs.parquet
Reading /home/reese/row-level-string-fingerprints/title_strs.parquet ...
Rows: 2,528,312
mode=prefix bits=8 -> buckets=256 -> column=q8_prefix
Sampling 500,000 rows to estimate 256 boundaries...
Assigning q8_prefix codes for all rows...
Saved boundaries: q8_prefix_boundaries.npy
Wrote q8_prefix_boundaries.txt
Wrote: title_strs_prefix_b8.parquet

q8_prefix distribution:
  non-empty buckets: 250 / 256
  max bucket size:   25,330
  min bucket size:   1

Top 10 buckets:
q8_prefix
70    25330
68    20960
56    19454
58    18587
61    18174
1     17836
59    17692
66    16082
15    15912
72    15839
Name: count, dtype: int64
Wrote q8_prefix_bucket_stats.csv
Wrote title_prefix_samples_b8.csv
+ /home/reese/row-level-string-fingerprints/.rowToBlock/bin/python build.py --bits 8 --suffix --input /home/reese/row-level-string-fingerprints/title_st

In [5]:
# Evaluation helpers. Prefix/suffix filtering is evaluated in DuckDB; substring
# filtering uses the project's Arrow scanner, which applies the bitmask first.
from bench_all_prefixes import bucket_range
from custom_scan import baseline_scan, fp_scan, load_features, pattern_mask

def sql_literal(value):
    return value.replace("'", "''")

def median_ms(thunk):
    for _ in range(WARMUP_RUNS):
        thunk()
    timings = []
    for _ in range(TIMED_REPETITIONS):
        start = time.perf_counter()
        thunk()
        timings.append((time.perf_counter() - start) * 1000)
    return float(np.median(timings))

def evaluate_prefix_or_suffix(item):
    mode, pattern = item.query_type, item.pattern
    parquet = PROJECT_DIR / f'title_strs_{mode}_b{PREFIX_BITS}.parquet'
    boundaries = np.load(PROJECT_DIR / f'q{PREFIX_BITS}_{mode}_boundaries.npy')
    code_col = f'q{PREFIX_BITS}_{mode}'
    lo, hi = bucket_range(pattern, boundaries, PREFIX_BITS, suffix=(mode == 'suffix'))
    like = f'%{sql_literal(pattern)}' if mode == 'suffix' else f'{sql_literal(pattern)}%'
    con = duckdb.connect()
    con.execute('PRAGMA threads=1')
    con.execute("CREATE TABLE t AS SELECT * FROM read_parquet(?)", [str(parquet)])
    con.execute(f'SELECT COUNT(*) FROM t').fetchone()  # warm file/table cache
    full_sql = f"SELECT COUNT(*) FROM t WHERE {TEXT_COLUMN} ILIKE '{like}'"
    fp_sql = f"SELECT COUNT(*) FROM t WHERE {code_col} BETWEEN {lo} AND {hi} AND {TEXT_COLUMN} ILIKE '{like}'"
    candidates_sql = f'SELECT COUNT(*) FROM t WHERE {code_col} BETWEEN {lo} AND {hi}'
    total = con.execute('SELECT COUNT(*) FROM t').fetchone()[0]
    matches = con.execute(full_sql).fetchone()[0]
    fp_matches = con.execute(fp_sql).fetchone()[0]
    candidates = con.execute(candidates_sql).fetchone()[0]
    baseline_ms, fingerprint_ms = median_ms(lambda: con.execute(full_sql).fetchone()), median_ms(lambda: con.execute(fp_sql).fetchone())
    con.close()
    return dict(bucket_lo=lo, bucket_hi=hi, candidate_rows=candidates, row_groups_skipped=np.nan,
                total_rows=total, match_count=matches, fingerprint_match_count=fp_matches,
                baseline_ms=baseline_ms, fingerprint_ms=fingerprint_ms)

def evaluate_substring(item):
    features, ngram = load_features()
    pf = pq.ParquetFile(PROJECT_DIR / 'title_strs_substr_fp16.parquet')
    mask = pattern_mask(item.pattern, features, ngram)
    base = baseline_scan(pf, item.pattern)
    guided = fp_scan(pf, mask, item.pattern)
    baseline_ms = median_ms(lambda: baseline_scan(pf, item.pattern))
    fingerprint_ms = median_ms(lambda: fp_scan(pf, mask, item.pattern))
    return dict(bucket_lo=np.nan, bucket_hi=np.nan, candidate_rows=np.nan, row_groups_skipped=guided[2],
                total_rows=pf.metadata.num_rows, match_count=base[1], fingerprint_match_count=guided[1],
                baseline_ms=baseline_ms, fingerprint_ms=fingerprint_ms, mask=mask, mask_bits=mask.bit_count())

In [6]:
results = []
for item in WORKLOAD.itertuples(index=False):
    print(f'Evaluating {item.name}: {item.query_type}({item.pattern!r})')
    metrics = evaluate_substring(item) if item.query_type == 'substring' else evaluate_prefix_or_suffix(item)
    results.append({**item._asdict(), **metrics})

results = pd.DataFrame(results)
results['equivalent'] = results['match_count'].eq(results['fingerprint_match_count'])
results['selectivity'] = results['match_count'] / results['total_rows']
results['candidate_fraction'] = results['candidate_rows'] / results['total_rows']
results['prune_rate'] = 1 - results['candidate_fraction']
results['speedup'] = results['baseline_ms'] / results['fingerprint_ms']

assert results['equivalent'].all(), 'Fingerprint filtering changed query results; inspect the workload and artifacts.'
display(results.sort_values(['query_type', 'name']))
results.to_csv(RESULTS_PATH, index=False)
print(f'Wrote {RESULTS_PATH}')

Evaluating prefix-the: prefix('the')
Evaluating suffix-ing: suffix('ing')
Evaluating contains-python: substring('python')


,name,query_type,pattern,bucket_lo,bucket_hi,candidate_rows,row_groups_skipped,total_rows,match_count,fingerprint_match_count,baseline_ms,fingerprint_ms,mask,mask_bits,equivalent,selectivity,candidate_fraction,prune_rate,speedup
0,prefix-the,prefix,the,217.0,236.0,198935.0,NaN,2528312,193218,193218,503.134904,74.632743,NaN,NaN,True,0.076422,0.078683,0.921317,6.741477
2,contains-python,substring,python,NaN,NaN,NaN,0.0,2528312,108,108,796.146188,305.561478,4132.0,3.0,True,0.000043,NaN,NaN,2.605519
1,suffix-ing,suffix,ing,139.0,141.0,29866.0,NaN,2528312,26801,26801,686.300832,72.696201,NaN,NaN,True,0.010600,0.011813,0.988187,9.440670


Wrote /home/reese/row-level-string-fingerprints/fingerprint_workload_results.csv


In [8]:
summary = (results.groupby('query_type', dropna=False)
           .agg(queries=('name', 'size'), matches=('match_count', 'sum'),
                median_baseline_ms=('baseline_ms', 'median'),
                median_fingerprint_ms=('fingerprint_ms', 'median'),
                median_speedup=('speedup', 'median'),
                mean_selectivity=('selectivity', 'mean'),
                mean_prune_rate=('prune_rate', 'mean'))
           .reset_index())
display(summary.style.format({
    'median_baseline_ms': '{:.2f}', 'median_fingerprint_ms': '{:.2f}',
    'median_speedup': '{:.2f}x', 'mean_selectivity': '{:.2%}', 'mean_prune_rate': '{:.2%}'
}))

# candidate_fraction/prune_rate are meaningful for prefix/suffix. For substring,
# use row_groups_skipped and the bitmask fields in the per-query table instead.

,query_type,queries,matches,median_baseline_ms,median_fingerprint_ms,median_speedup,mean_selectivity,mean_prune_rate
0,prefix,1,193218,503.13,74.63,6.74x,7.64%,92.13%
1,substring,1,108,796.15,305.56,2.61x,0.00%,nan%
2,suffix,1,26801,686.30,72.70,9.44x,1.06%,98.82%
